# 00: Synthetic cascade dataset

This notebook creates the small **supervised** dataset used by the Deep Learning notebooks. 

Graph instances are split **before** transitions are generated, so states from the same topology cannot appear in both training and test data.
This allows to maximize the training data generation efficience while protecting against **data leakage**.

The simulator is intentionally compact since it is not the focus of the course. It's optimization includes: the creation of infrastructure-like graphs, application of an initial failure, redistribution of failed load, and recording of one-step transitions.

In [ ]:
from __future__ import annotations

# Python Environment Packages
import json
import random
from dataclasses import dataclass # Allows to define structured state objects without writing large classes manually 
from pathlib import Path

# ML related Packages
import matplotlib.pyplot as plt
import networkx as nx   # Used to create and work with graphs
import numpy as np
import torch    # Main ML tool used in this project
import yaml  # Used to create a project configuration file 


## Project paths and configuration

In [ ]:
ROOT = Path.cwd().resolve() # Deifine the root of the project 
CONFIG_PATH = ROOT / "config.yaml"  # Locate the configuration file which is used to define how the project is ran

if not CONFIG_PATH.exists():    # Path mistake control
    raise FileNotFoundError(
        "config.yaml was not found. Run the notebook from the DL_project folder."
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as file:  # Read the configuration file
    config = yaml.safe_load(file)

DATA_DIR = ROOT / config["data"]["output_dir"]  # Find the data directory, create one if not existing yet
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Dataset folder:", DATA_DIR)


In this project a configuration file is used, an improvement that was implemented in the later stages of programming. 

A configuration file allows to separate the parameters used during training, creation of data, definition of the architecture, etc.. as an external file. Separating this allows for increased reproducibility and tracking of the project configuration, while also allowing easier parameter tuning.

In [ ]:
SEED = 17   # Define a seed for reproducibility of results
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)


## Graph and cascade state

In [ ]:
"""
'dataclass' is a more efficient version of the 'class' method for containing data.
The idea is that CascadeState will be a continer of various arrays of data, representing the full state
of the cascade at each simulation step.
"""
@dataclass
class CascadeState:
    load: np.ndarray    # current load of the node
    capacity: np.ndarray    # maximum supported load of the node
    failed: np.ndarray  # whether that node has failed
    protected: np.ndarray   # whether that nod has protection
    newly_failed: np.ndarray    # whether the node failed during the last transition
    dependency_group: np.ndarray    # synthetic functional group used for long-range dependencies
    step_index: int = 0 # current simulation step

    # The purpose of this function is to create a callable state object, useful to get the state of the cascade simply by calling it
    def copy(self) -> "CascadeState":  # This method returns another CascadeState object
        return CascadeState(
            load=self.load.copy(),
            capacity=self.capacity.copy(),
            failed=self.failed.copy(),
            protected=self.protected.copy(),
            newly_failed=self.newly_failed.copy(),
            dependency_group=self.dependency_group.copy(),
            step_index=self.step_index,
        )

# This class stores the parameters passed from the configuration file, its field cannot be changed
@dataclass(frozen=True)
class CascadeParameters:
    protection_alpha: float
    redistribution_fraction: float
    long_range_strength: float
    max_steps: int
    collapse_threshold: float


## Graph generation

Synthetic network creation

In [ ]:
"""
This Cell defines the functions that will be used to generate connected graphs
"""

def connect_components(graph: nx.Graph, rng: np.random.Generator) -> nx.Graph:
    
    graph = nx.convert_node_labels_to_integers(graph)   # Convert the data to integer, important as it will be used as labels for numpy arrays
    components = [list(component) for component in nx.connected_components(graph)]  # List comprehension to find the disconnected components
    
    for left, right in zip(components[:-1], components[1:]):    # function used to connect consecutive components
        graph.add_edge( # randomly choses the nodes to connect among the disconnected ones
            int(rng.choice(left)), 
            int(rng.choice(right))
        )
    graph.remove_edges_from(nx.selfloop_edges(graph))   # remove eventual self connected loops
    return graph

"""
The graphs generation supports three graphs families:
-Erdős–Rényi: Every possible edge is independently present with probability p. p is sampled between 2.8/N and 4.8/N.
-Barabási–Albert: New nodes preferentially connect to already well-connected nodes.
-Watts–Strogatz: Tends to create a small-world network, high clusterning and short paths.

This is done to create models able to generalize well and transfer their skills across network structures, 
similarly to what we would exepct in a real world scenario.
"""
def generate_graph(family: str, num_nodes: int, rng: np.random.Generator) -> nx.Graph:
    seed = int(rng.integers(0, 2**31 - 1))  # The generation of datasets requires a different seed for each generation so to not replicate at creating the same structure each time
    if family == "er":
        probability = float(rng.uniform(2.8 / num_nodes, 4.8 / num_nodes))
        graph = nx.erdos_renyi_graph(num_nodes, probability, seed=seed)
    elif family == "ba":
        edges_per_new_node = int(rng.integers(2, min(5, num_nodes - 1)))    # The higher the num_nodes the higher the maximum number of links a node can get becomes
        graph = nx.barabasi_albert_graph(num_nodes, edges_per_new_node, seed=seed)
    elif family == "ws":
        neighbours = min(int(rng.choice([4, 6])), num_nodes - 1)
        neighbours -= neighbours % 2    # be sure that the number of neighbours is even
        graph = nx.watts_strogatz_graph(
            num_nodes, neighbours, float(rng.uniform(0.08, 0.30)), seed=seed
        )
    else:
        raise ValueError(f"Unknown graph family: {family}")
    return connect_components(graph, rng)


## Cascade dynamics

The parameter `redistribution_fraction` is the fraction \(\rho\) of a failed node's load that remains in the system and is redistributed. Any remaining fraction \(1-\rho\) is interpreted as shed or unserved load.

In the long-range regime, `long_range_strength` is the fraction \(\gamma\) of the redistributed load that follows the functional dependency channel. Therefore, when both local and dependency targets are available,

\[
L_i^{\mathrm{local}}=(1-\gamma)\rho\ell_i,
\qquad
L_i^{\mathrm{dependency}}=\gamma\rho\ell_i.
\]

The two contributions sum to \(\rho\ell_i\), so the long-range mechanism does not create additional load. Dependency targets are active nodes in the same synthetic functional group, excluding direct graph neighbours so that a node is not charged twice through both channels.


In [ ]:
def initialize_state(graph: nx.Graph, rng: np.random.Generator, data_cfg: dict) -> CascadeState:
    n = graph.number_of_nodes()
    load = rng.uniform(*data_cfg["load_range"], size=n).astype(np.float32)   # every node starts with a load between 0.65 and 1.0. The * is used to unpack a sequence into separate arguments and pass the two values as upper and lower bonds
    tolerance = rng.uniform(*data_cfg["tolerance_range"], size=n).astype(np.float32)    # tolerance is random between 0.1 and 0.3 (with the current setup)
    capacity = ((1.0 + tolerance) * load).astype(np.float32)    # capacity is load + tolerance
    num_groups = int(data_cfg["long_range"]["num_groups"])  # reads how many synthetic functional dependency groups should exist

    return CascadeState(    # initialize the state given the configuration parameters
        load=load,
        capacity=capacity,
        failed=np.zeros(n, dtype=np.float32),
        protected=np.zeros(n, dtype=np.float32),
        newly_failed=np.zeros(n, dtype=np.float32),
        dependency_group=rng.integers(0, num_groups, size=n, dtype=np.int64),  # randomly assigns each node to one functional dependency group
    )


# we artificially generate an initial shock that manually fails up until k nodes
def apply_initial_shock(state: CascadeState, graph: nx.Graph, rng: np.random.Generator, k: int, shock_type: str) -> np.ndarray:
    n = graph.number_of_nodes()
    k = max(1, min(k, n))

    degree = np.asarray([graph.degree(i) for i in range(n)], dtype=np.float32)  # obtains an array defining containing the degree for each node

    # There are different shock possibilities, depending on the type of system you want to simulate
    if shock_type == "mixed":   # randomly chooses between the other methods
        shock_type = str(rng.choice(["random", "degree", "load"], p=[0.6, 0.2, 0.2]))
    if shock_type == "random":  # choses randomly
        nodes = rng.choice(n, size=k, replace=False)
    elif shock_type == "degree":    # fails highest degree nodes
        nodes = np.argsort(-degree)[:k]
    elif shock_type == "load":  # fails the node with the highest load
        nodes = np.argsort(-state.load)[:k]
    else:
        raise ValueError(f"Unknown shock type: {shock_type}")
    
    state.failed[nodes] = 1.0   # flags the nodes as failed
    state.newly_failed[nodes] = 1.0
    return np.asarray(nodes, dtype=np.int64)


# Now the nodes have failed the next step is to redistribute the load to the neighbouring nodes
def cascade_step(graph: nx.Graph, state: CascadeState, params: CascadeParameters) -> CascadeState:
    # rho is the total fraction of failed load that remains in the system;
    # gamma is the fraction of that redistributed load assigned to the dependency channel.
    if not 0.0 <= params.redistribution_fraction <= 1.0:
        raise ValueError("redistribution_fraction (rho) must be between 0 and 1.")
    if not 0.0 <= params.long_range_strength <= 1.0:
        raise ValueError("long_range_strength (gamma) must be between 0 and 1.")

    next_state = state.copy()   # makes a copy of the state
    active = state.failed < 0.5 # create a mask to verify the active nodes
    sources = np.flatnonzero(state.newly_failed > 0.5)  # these are the nodes that have just failed whose effect must be propagated

    for source in sources:
        source_load = float(state.load[source])
        local_targets = [target for target in graph.neighbors(int(source)) if active[target]]   # selects the targets (nodes to which the initial load will be propagated) choosing among the Nearest Neighbours that are still active

        dependency_targets = np.asarray([], dtype=np.int64)
        if params.long_range_strength > 0.0:    # in the long-range regime, some redistributed load follows functional dependencies
            same_group = np.flatnonzero(active & (state.dependency_group == state.dependency_group[source]))    # active nodes belonging to the same synthetic dependency group
            same_group = same_group[same_group != source]   # remove source

            # Remove direct neighbours from the dependency targets so that the same node
            # does not receive load twice through both the local and dependency channels.
            local_target_set = set(local_targets)
            dependency_targets = np.asarray(
                [target for target in same_group if target not in local_target_set],
                dtype=np.int64,
            )

        redistributed_load = params.redistribution_fraction * source_load   # rho * source_load; the remaining fraction is considered shed or unserved
        has_local_targets = len(local_targets) > 0
        has_dependency_targets = len(dependency_targets) > 0

        # If both channels are available, gamma splits the same conserved redistributed load.
        # If only one channel is available, all transferable load is rerouted through that channel.
        if has_local_targets and has_dependency_targets:
            dependency_load = params.long_range_strength * redistributed_load
            local_load = redistributed_load - dependency_load
        elif has_local_targets:
            local_load = redistributed_load
            dependency_load = 0.0
        elif has_dependency_targets:
            local_load = 0.0
            dependency_load = redistributed_load
        else:
            local_load = 0.0
            dependency_load = 0.0   # no active destination exists, so the transferable load is lost

        if has_local_targets:   # divide the local part equally among active graph neighbours
            local_share = local_load / len(local_targets)
            next_state.load[local_targets] += local_share

        if has_dependency_targets:   # divide the dependency part equally among active same-group nodes
            dependency_share = dependency_load / len(dependency_targets)
            next_state.load[dependency_targets] += dependency_share

        next_state.load[source] = 0.0

    effective_capacity = state.capacity * np.where( # here we apply the protection, where we multiply the capacity with the parameter alpha, useful for later application of this project
        state.protected > 0.5, params.protection_alpha, 1.0
    )
    newly_failed = ((next_state.load > effective_capacity) & active).astype(np.float32) # a state failes if the enhanced capacity is less than the new total load
    next_state.failed = np.maximum(state.failed, newly_failed)  # update the failed nodes
    next_state.newly_failed = newly_failed
    next_state.step_index = state.step_index + 1
    return next_state

# define when the cascade stops
def is_terminal(state: CascadeState, params: CascadeParameters) -> bool:
    return bool(
        state.newly_failed.sum() == 0   # no new node has failed
        or state.step_index >= params.max_steps # maximum number of steps is reached
        or state.failed.mean() >= params.collapse_threshold # we have superated the threshold of failed nodes
    )   # returns a boolean value, 1 if terminal

# function to repeatedly apply cascade_step until temrination
def rollout_to_end(graph: nx.Graph, state: CascadeState, params: CascadeParameters) -> CascadeState:
    current = state.copy()
    while not is_terminal(current, params):
        current = cascade_step(graph, current, params)
    return current


## Features and sample format

The simulator state needs to be converted into Neural Network inputs and targets.

### Laplacian positional encoding

The node features describe the current state of each node, but they do not fully describe its position in the graph.

For this reason, we compute the normalized graph Laplacian and use some of its eigenvectors as positional encodings. These vectors give each node additional structural coordinates, helping the model distinguish nodes with similar features but different roles or locations in the network.


In [ ]:
# Convert the graph structure into Laplacian positional encodings.
def normalized_laplacian_pe(graph: nx.Graph, dim: int) -> np.ndarray:
    n = graph.number_of_nodes()
    adjacency = nx.to_numpy_array(graph, nodelist=range(n), dtype=np.float64)   # get the adjacency matrix
    degree = adjacency.sum(axis=1)  # degree of the graph
    inv_sqrt = np.zeros_like(degree)
    inv_sqrt[degree > 0] = degree[degree > 0] ** -0.5   # degree matrix already modified to be used in the formula of the graph laplacian: L=I−D^{-1/2}*A*D^{−1/2}
    laplacian = np.eye(n) - inv_sqrt[:, None] * adjacency * inv_sqrt[None, :]   # Obtain the laplacian
    _, eigenvectors = np.linalg.eigh(laplacian) # The eigenvector of the Laplacian describe large-scale graph structure
    vectors = eigenvectors[:, 1:1 + dim]    # skips the first trivial evect which would only complicate training
    # the obtained evect become positional encodings which will be used by the ransformer
    if vectors.shape[1] < dim:
        vectors = np.pad(vectors, ((0, 0), (0, dim - vectors.shape[1])))    # since we removed the first eigenvector and the model expects dim positional features, we add a padding to have consistent feature dimension
    for column in range(vectors.shape[1]):  # this step is necessary to check that the NN does not misintepretate the eigenvlues since ev = -ev so we fi this
        pivot = int(np.argmax(np.abs(vectors[:, column])))  # looks for the largest absolute vlaue of eigenvectors
        if vectors[pivot, column] < 0:
            vectors[:, column] *= -1    # puts the absolute largest eigenvector positive 
    return vectors.astype(np.float32)


def shortest_path_matrix(graph: nx.Graph, max_distance: int) -> np.ndarray:
    n = graph.number_of_nodes()
    matrix = np.full((n, n), max_distance + 1, dtype=np.int64)  # create the matrix object
    np.fill_diagonal(matrix, 0)

    for source, lengths in nx.all_pairs_shortest_path_length(graph, cutoff=max_distance):
        for target, distance in lengths.items():
            matrix[source, target] = distance   # for each pair define the shortest-path idstance between the two nodes
    return matrix

# define the node features to give to the model
def node_features(graph: nx.Graph, state: CascadeState, num_groups: int) -> np.ndarray:
    n = graph.number_of_nodes()
    degree = np.asarray([graph.degree(i) for i in range(n)], dtype=np.float32)
    degree /= max(float(degree.max()), 1.0) # degree fraction of each node
    load_ratio = state.load / np.maximum(state.capacity, 1e-6)
    action_flag = np.zeros(n, dtype=np.float32)  # reserved for the later intervention model
    
    group_one_hot = np.eye(num_groups, dtype=np.float32)[state.dependency_group]    # dependency group represented by one hot encoding
    base = np.column_stack([
        state.load, state.capacity, load_ratio, state.failed,
        state.protected, action_flag, degree,
    ]).astype(np.float32)
    return np.concatenate([base, group_one_hot], axis=1)


# creates one complete learning sample
def state_to_sample(graph, state, next_state, final_state, family, regime, graph_id, data_cfg):
    n = graph.number_of_nodes()
    adjacency = nx.to_numpy_array(graph, nodelist=range(n), dtype=np.float32)
    num_groups = int(data_cfg["long_range"]["num_groups"])

    return {
        "x": torch.from_numpy(node_features(graph, state, num_groups)),
        "adj": torch.from_numpy(adjacency),
        "lap_pe": torch.from_numpy(normalized_laplacian_pe(graph, int(data_cfg["lap_pe_dim"]))),
        "dist": torch.from_numpy(shortest_path_matrix(graph, int(data_cfg["max_distance"]))),
        "y_node": torch.from_numpy(next_state.failed.astype(np.float32)),   # targets node load regression
        "y_load": torch.from_numpy(next_state.load.astype(np.float32)),     # nodee failure classification target
        "y_graph": torch.tensor(float(final_state.failed.mean()), dtype=torch.float32), # graph level cascade regression target
        "num_nodes": n,
        "graph_id": graph_id,
        "family": family,
        "regime": regime,
    }


## Build graph-disjoint splits

In [ ]:
def build_dataset(config: dict) -> dict[str, list[dict]]:
    data_cfg = config["data"]
    local_rng = np.random.default_rng(int(config["seed"]))  # need a random personal seed
    num_graphs = int(data_cfg["num_graphs"])    # the number of graphs is defined in the yaml document
    split = data_cfg["split"]   # percentage of dataset that goes in each validation training and test dataset

    graph_ids = np.arange(num_graphs)
    local_rng.shuffle(graph_ids)
    train_end = int(num_graphs * split[0])  # integer defining the index until we have training dataser
    val_end = train_end + int(num_graphs * split[1])
    split_map = {int(i): "train" for i in graph_ids[:train_end]}
    split_map.update({int(i): "val" for i in graph_ids[train_end:val_end]})
    split_map.update({int(i): "test" for i in graph_ids[val_end:]})

    samples = {"train": [], "val": [], "test": []}
    families = list(data_cfg["graph_families"]) # considers the three possible families
    regimes = list(data_cfg["regimes"]) # local or long range

    for graph_id in range(num_graphs):
        # Cycling gives every mechanism repeated coverage; the graph itself remains random.
        family = families[graph_id % len(families)] # cycles through the families
        regime = regimes[(graph_id // len(families)) % len(regimes)]

        graph = generate_graph(family, int(data_cfg["num_nodes"]), local_rng)   # generate topology
        state = initialize_state(graph, local_rng, data_cfg)    # initializes the state
        shock_cfg = data_cfg["shock"]   # applies a shock
        shock_size = int(local_rng.integers(int(shock_cfg["min_k"]), int(shock_cfg["max_k"]) + 1))
        apply_initial_shock(state, graph, local_rng, shock_size, str(shock_cfg["type"]))

        params = CascadeParameters( # creates the cascade parameters
            protection_alpha=float(data_cfg["protection_alpha"]),
            redistribution_fraction=float(data_cfg["redistribution_fraction"]),
            long_range_strength=float(data_cfg["long_range"]["strength"]) if regime == "long_range" else 0.0,
            max_steps=int(data_cfg["max_steps"]),
            collapse_threshold=float(data_cfg["collapse_threshold"]),
        )

        transitions = 0
        while transitions < int(data_cfg["transitions_per_graph"]): # simulate up to a chosen number of transition (after splitting the dataset)
            next_state = cascade_step(graph, state, params)
            final_state = rollout_to_end(graph, next_state, params)
            samples[split_map[graph_id]].append(
                state_to_sample(graph, state, next_state, final_state, family, regime, graph_id, data_cfg)
            )
            transitions += 1
            state = next_state
            if is_terminal(state, params):
                break

    return samples


def save_dataset(samples: dict[str, list[dict]], config: dict, output_dir: Path) -> dict:
    for split_name, split_samples in samples.items():
        torch.save(split_samples, output_dir / f"{split_name}.pt")  # optimal way to save and load for this dataset
    all_samples = sum(samples.values(), [])
    metadata = {    # also saves the metadata, might be useful for training
        "feature_dim": int(all_samples[0]["x"].shape[1]),
        "lap_pe_dim": int(all_samples[0]["lap_pe"].shape[1]),
        "max_nodes": max(int(sample["num_nodes"]) for sample in all_samples),
        "split_sizes": {name: len(values) for name, values in samples.items()},
        "graph_ids": {
            name: sorted({int(sample["graph_id"]) for sample in values})
            for name, values in samples.items()
        },
        "feature_names": [
            "load", "capacity", "load_ratio", "failed", "protected",
            "action_flag", "degree_norm", "dependency_group_0",
            "dependency_group_1", "dependency_group_2",
        ],
    }
    (output_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return metadata


## Generate and inspect

In [ ]:
FORCE_REBUILD = True    # this is made so that every notebook run rebuilds and overwrites the old dataset (True while debugging)
required_files = [DATA_DIR / f"{split}.pt" for split in ["train", "val", "test"]]   # define the path of the files
if FORCE_REBUILD or not all(path.exists() for path in required_files):  # rebuild also if something is missing
    dataset = build_dataset(config)
    metadata = save_dataset(dataset, config, DATA_DIR)
else:
    dataset = {split: torch.load(DATA_DIR / f"{split}.pt", weights_only=False) for split in ["train", "val", "test"]}   # load
    metadata = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))

print(json.dumps(metadata, indent=2))


In [ ]:
# Leakage checks to check that there is no repetition of the same id in more than one dataset (this would correspond to data contamination)
train_ids = set(metadata["graph_ids"]["train"])
val_ids = set(metadata["graph_ids"]["val"])
test_ids = set(metadata["graph_ids"]["test"])
assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)
# assert metadata["feature_dim"] == 10
print("Graph-disjoint split check passed.")


In [ ]:
# inspect one sample
sample = dataset["train"][0]
for key, value in sample.items():
    if torch.is_tensor(value):
        print(f"{key:10s}", tuple(value.shape), value.dtype)
    else:
        print(f"{key:10s}", value)


## One graph-state visualization

In [ ]:
from matplotlib.patches import Patch

adjacency = sample["adj"].numpy()
graph = nx.from_numpy_array(adjacency) # convert the pythorch tensor back to graph

failed_now = sample["x"][:, 3].numpy()
failed_next = sample["y_node"].numpy()

node_values = failed_now + failed_next

colors = []

for value in node_values:
    if value == 0:
        colors.append("blue")
    elif value == 1:
        colors.append("green")
    else:
        colors.append("red")

plt.figure(figsize=(7, 5))

position = nx.spring_layout(graph, seed=SEED)

nx.draw_networkx(
    graph,
    position,
    node_size=350,
    with_labels=True,
    node_color=colors,
    edge_color="lightgray",
)

legend = [
    Patch(color="blue", label="Healthy"),
    Patch(color="green", label="Fails next / one failure flag"),
    Patch(color="red", label="Failed now and next"),
]

plt.legend(handles=legend)
plt.title("Node labels are IDs; colours represent failure state")
plt.axis("off")
plt.tight_layout()
plt.show()
